In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

# Load dataset - REPLACE THE PATH WITH THE CORRECT ONE IF IT'S DIFFERENT
df = pd.read_csv("/content/complete_labeled_dataset - complete_labeled_dataset.csv.csv")
df.head()

,headline,label,importance,binary_label,confidence,source,label_name
0,"Markets Close Mostly Lower Again; ROST, PANW R...",Negative,major,1,1.0,original_labeled,major
1,"Gap plummets on earnings miss, cuts full-year ...",Negative,major,1,1.0,original_labeled,major
2,Billionaire Ken Fisher is Selling These 10 Stocks,Negative,non-major,0,1.0,original_labeled,non-major
3,"Corning net income drops 13%, shares fall",Negative,non-major,0,1.0,original_labeled,non-major
4,Internet Explorer shutdown to cause Japan prob...,Negative,non-major,0,1.0,original_labeled,non-major


In [7]:
df = df[["headline", "label", "label_name"]]

df = df.rename(columns={
    "label": "sentiment",
    "label_name": "importance"
})

df = df.dropna()


In [8]:
le_sentiment = LabelEncoder()
le_importance = LabelEncoder()

df["sentiment_encoded"] = le_sentiment.fit_transform(df["sentiment"])
df["importance_encoded"] = le_importance.fit_transform(df["importance"])


In [9]:
X = df["headline"]
y = df[["sentiment_encoded", "importance_encoded"]]

df["strat_key"] = df["sentiment"] + "_" + df["importance"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=df["strat_key"]
)


In [10]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", MultiOutputClassifier(MultinomialNB()))
])


In [11]:
param_grid = {
    "tfidf__ngram_range": [(1,1), (1,2)],
    "tfidf__min_df": [2, 5],
    "tfidf__max_df": [0.9, 1.0],
    "clf__estimator__alpha": [0.1, 0.5, 1.0]
}

grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)


Fitting 3 folds for each of 24 candidates, totalling 72 fits


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:1108: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan nan
 nan nan nan nan nan nan]
  warnings.warn(


GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('tfidf', TfidfVectorizer()),
                                       ('clf',
                                        MultiOutputClassifier(estimator=MultinomialNB()))]),
             n_jobs=-1,
             param_grid={'clf__estimator__alpha': [0.1, 0.5, 1.0],
                         'tfidf__max_df': [0.9, 1.0], 'tfidf__min_df': [2, 5],
                         'tfidf__ngram_range': [(1, 1), (1, 2)]},
             scoring='accuracy', verbose=2)

In [12]:
print("Best Parameters:")
print(grid.best_params_)

best_model = grid.best_estimator_


Best Parameters:
{'clf__estimator__alpha': 0.1, 'tfidf__max_df': 0.9, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 1)}


In [13]:
y_pred = best_model.predict(X_test)

# Decode
y_pred_sent = le_sentiment.inverse_transform(y_pred[:,0])
y_pred_imp  = le_importance.inverse_transform(y_pred[:,1])

y_true_sent = le_sentiment.inverse_transform(y_test.iloc[:,0])
y_true_imp  = le_importance.inverse_transform(y_test.iloc[:,1])

print("=== SENTIMENT ===")
print(classification_report(y_true_sent, y_pred_sent))
print("Accuracy:", accuracy_score(y_true_sent, y_pred_sent))

print("\n=== IMPORTANCE ===")
print(classification_report(y_true_imp, y_pred_imp))
print("Accuracy:", accuracy_score(y_true_imp, y_pred_imp))


=== SENTIMENT ===
              precision    recall  f1-score   support

    Negative       0.69      0.58      0.63       961
     Neutral       0.68      0.58      0.62      1749
    Positive       0.69      0.80      0.74      2490

    accuracy                           0.68      5200
   macro avg       0.68      0.65      0.66      5200
weighted avg       0.68      0.68      0.68      5200

Accuracy: 0.6844230769230769

=== IMPORTANCE ===
              precision    recall  f1-score   support

       major       0.89      0.77      0.82       975
   non-major       0.95      0.98      0.96      4225

    accuracy                           0.94      5200
   macro avg       0.92      0.87      0.89      5200
weighted avg       0.94      0.94      0.94      5200

Accuracy: 0.9388461538461539


In [14]:
def predict_headline_nb(texts):
    preds = best_model.predict(texts)
    return pd.DataFrame({
        "headline": texts,
        "sentiment": le_sentiment.inverse_transform(preds[:,0]),
        "importance": le_importance.inverse_transform(preds[:,1])
    })

predict_headline_nb([
    "Market falls after unexpected inflation data",
    "Tech stocks show strong recovery in Q3"
])


,headline,sentiment,importance
0,Market falls after unexpected inflation data,Negative,non-major
1,Tech stocks show strong recovery in Q3,Positive,non-major
